# Comparison of Generative Data Sampling

Code is from [This Github Repo](https://github.com/Diyago/Tabular-data-generation/tree/master) to save time

basically will be running all four options and comparing outputs to show how these vary and if there is noticable difference for the quality of the data and other implications for synthetic data to improve dataset size (like if that causes bias?)


In [ ]:
import kagglehub
import pandas as pd
import ipaddress
from sklearn.preprocessing import LabelEncoder

# Download latest version to compare with the two generated datasets from models
path = kagglehub.dataset_download(
    "munaalhawawreh/xiiotid-iiot-intrusion-dataset")

print("Path to dataset files:", path)
print(path)
df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


# 1. Basic Cleaning: Strip whitespace from columns and string values
df.columns = df.columns.str.strip()
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Universal IP & Port Normalizer (Handles IPv4, IPv6, ?, -, and Ports)


def universal_to_numeric(val):
    val = str(val).strip()
    if val in ['?', '-', '', 'None']:
        return 0
    try:
        # If it looks like an IP, convert to int
        if '.' in val or ':' in val:
            return int(ipaddress.ip_address(val))
        # Otherwise, treat as a standard number
        return int(float(val))
    except:
        return 0


for col in ['Scr_IP', 'Des_IP', 'Scr_port', 'Des_port']:
    df[col] = df[col].apply(universal_to_numeric)

# 3. Timestamp Fix (Convert Unix to Numeric safely)
df['Timestamp'] = pd.to_numeric(
    df['Timestamp'], errors='coerce').fillna(0).astype(int)
df['Time'] = pd.to_datetime(df['Timestamp'], unit='s')
df.drop(columns=['Timestamp'], inplace=True)
# 4. Boolean/Flag Mapping (TRUE/FALSE strings or actual Bools -> 0/1)
bool_cols = [
    'is_syn_only', 'Is_SYN_ACK', 'is_pure_ack', 'is_with_payload',
    'FIN or RST', 'Bad_checksum', 'is_SYN_with_RST', 'anomaly_alert'
]
for col in bool_cols:
    if col in df.columns:
        # Map strings if they exist, otherwise force to int
        if df[col].dtype == 'object':
            df[col] = df[col].str.upper().map(
                {'TRUE': 1, 'FALSE': 0, '-': 0, '?': 0})
        df[col] = df[col].fillna(0).astype(int)

# 5. Mass Numeric Conversion for Metrics (Duration, Bytes, CPU stats, etc.)
# We replace '?' and '-' with 0 first
df.replace(['?', '-'], 0, inplace=True)
cat_cols = ['Protocol', 'Service', 'class1', 'class2', 'class3']
encoders = {}
# 6. Categorical Cleanup
for col in ['Protocol', 'Service']:
    df[col] = df[col].replace(0, 'unknown').astype(str)
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        # Ensure everything is a string before encoding to avoid mixed-type errors
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

# 3. Final safety check: ensure NO strings remain in the numeric features
# (This prevents the 'udp' error once and for all)
for col in df.columns:
    if df[col].dtype == 'object':
        # If any stray strings exist, force them to numeric or drop
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
# Identify all columns that should be numeric (excluding categorical labels)

numeric_cols = [c for c in df.columns if c not in cat_cols + ['Date']]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 7. Final Drop
df.drop(columns=['Date'], inplace=True)

df.info()

Path to dataset files: C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1
C:\Users\antho\.cache\kagglehub\datasets\munaalhawawreh\xiiotid-iiot-intrusion-dataset\versions\1


C:\Users\antho\AppData\Local\Temp\ipykernel_24972\1249797039.py:12: DtypeWarning: Columns (0: Timestamp, 1: Scr_port, 2: Des_port, 3: missed_bytes, 4: anomaly_alert) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path + "\\X-IIoTID dataset.csv", skipinitialspace=True)


In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "svg"
target_cols = ['class1', 'class2', 'class3']

X = df.drop(columns=target_cols)
y = df[['class1']]  # Focus on class1 for now

# Create the 1,000 row set
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X, y, train_size=800, test_size=200, stratify=y, random_state=42
)

X_train_med, X_test_med, y_train_med, y_test_med = train_test_split(
    X, y, train_size=8000, test_size=2000, stratify=y, random_state=42
)

# Create the 10,000 row set
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X, y, train_size=80000, test_size=20000, stratify=y, random_state=42
)

labels = encoders['class1'].classes_

dataset_names = ['Original', 'Small', 'Medium', 'Large']


def get_dist(series, labels):
    dist = series.value_counts(normalize=True).reindex(
        range(len(labels)), fill_value=0)
    dist.index = labels
    return dist


original_dist = get_dist(df['class1'], labels)
small_test_dist = get_dist(y_test_small['class1'], labels)
med_test_dist = get_dist(y_test_med['class1'], labels)
large_test_dist = get_dist(y_test_large['class1'], labels)

# Group distributions for iteration
all_dists = [original_dist, small_test_dist, med_test_dist, large_test_dist]

threshold = 0.01
common_classes = original_dist[original_dist > threshold].index
rare_classes = original_dist[original_dist <= threshold].index

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Common Classes (>1%)",
                                    "Rare Classes (<1%)"),
                    vertical_spacing=0.15)

# Add Common Classes to Legend
for cls in common_classes:
    y_vals = [d[cls] for d in all_dists]
    fig.add_trace(go.Bar(
        x=dataset_names,
        y=y_vals,
        name=str(cls),
        legendgroup="common"  # Optional: groups them in the legend
    ), row=1, col=1)

# Add Rare Classes to Legend
for cls in rare_classes:
    y_vals = [d[cls] for d in all_dists]
    fig.add_trace(go.Bar(
        x=dataset_names,
        y=y_vals,
        name=str(cls),
        legendgroup="rare"
    ), row=2, col=1)

# 5. Update Layout
fig.update_layout(
    width=1000,
    height=800,
    barmode='group',  # Set to 'stack' if you want to see total composition
    title="Class Distribution Across Split Scales",
    xaxis_title="Dataset Split",
    yaxis_title="Proportion (0.0 - 1.0)",
    legend_title="Class Labels",
    showlegend=True
)

fig.write_image("Data_Distribution_Legend.svg")
fig.show()

# --- Table Printing Logic ---
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # Ensure the width is large enough

orig_counts = df['class1'].value_counts()
orig_pcts = df['class1'].value_counts(normalize=True) * 100

# 2. Calculate counts and percentages for the Test Set
small_test_counts = y_test_small['class1'].value_counts()
small_test_pcts = y_test_small['class1'].value_counts(normalize=True) * 100
med_test_dist = y_test_med['class1'].value_counts()
med_test_pcts = y_test_med['class1'].value_counts(normalize=True) * 100
large_test_counts = y_test_large['class1'].value_counts()
large_test_pcts = y_test_large['class1'].value_counts(normalize=True) * 100

# 3. Combine into a single detailed DataFrame
comparison_table = pd.DataFrame({
    'Original (count)': orig_counts,
    'Original (%)': orig_pcts,
    'Small Test Set (count)': small_test_counts,
    'Small Test Set (%)': small_test_pcts,
    'Medium Test Set (count)': med_test_dist,
    'Medium Test Set (%)': med_test_pcts,
    'Large Test Set (count)': large_test_counts,
    'Large Test Set (%)': large_test_pcts
})


labels = encoders['class1'].classes_
comparison_table.index = [labels[i] for i in comparison_table.index]

# 5. Clean up the display
print("--- Detailed Distribution Table (class1) ---")
print(comparison_table.round(5))

In [ ]:
from tabgan.sampler import GANGenerator, ForestDiffusionGenerator, LLMGenerator

categorical_features = ['Protocol', 'Service']


small_GAN_train, small_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 200, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_small, target=y_train_small, test_df=X_test_small)
med_GAN_train, med_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 500, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_med, target=y_train_med, test_df=X_test_med)
large_GAN_train, large_GAN_target = GANGenerator(gen_params={
    'cat_cols': categorical_features, "deep_copy": True, "only_generated_data": True,
    "batch_size": 500, "patience": 25, "epochs": 500, "gen_x_times": 2}) \
    .generate_data_pipe(train_df=X_train_large, target=y_train_large, test_df=X_test_large)

In [ ]:
print(len(small_GAN_train), len(small_GAN_target),
      len(large_GAN_train), len(large_GAN_target))


labels = encoders['class1'].classes_


small_test_dist = get_dist(small_test_dist, labels)
med_test_dist = get_dist(med_test_dist, labels)
large_test_dist = get_dist(large_test_dist, labels)

# Group distributions for iteration
all_dists = [original_dist, small_test_dist, med_test_dist, large_test_dist]


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Common Classes (>1%)",
                                    "Rare Classes (<1%)"),
                    vertical_spacing=0.15)

# Add Common Classes to Legend
for cls in common_classes:
    y_vals = [d[cls] for d in all_dists]
    fig.add_trace(go.Bar(
        x=dataset_names,
        y=y_vals,
        name=str(cls),
        legendgroup="common"  # Optional: groups them in the legend
    ), row=1, col=1)

# Add Rare Classes to Legend
for cls in rare_classes:
    y_vals = [d[cls] for d in all_dists]
    fig.add_trace(go.Bar(
        x=dataset_names,
        y=y_vals,
        name=str(cls),
        legendgroup="rare"
    ), row=2, col=1)

# 5. Update Layout
fig.update_layout(
    width=1000,
    height=800,
    barmode='group',  # Set to 'stack' if you want to see total composition
    title="Class Distribution Across Split Scales",
    xaxis_title="Dataset Split",
    yaxis_title="Proportion (0.0 - 1.0)",
    legend_title="Class Labels",
    showlegend=True
)

fig.write_image("Data_Distribution_Legend.svg")
fig.show()

small_test_counts = pd.Series(small_GAN_target).value_counts()
small_test_pcts = pd.Series(
    small_GAN_target).value_counts(normalize=True) * 100
med_test_dist = pd.Series(med_GAN_target).value_counts()
med_test_pcts = pd.Series(
    med_GAN_target).value_counts(normalize=True) * 100
large_test_counts = pd.Series(large_GAN_target).value_counts()
large_test_pcts = pd.Series(
    large_GAN_target).value_counts(normalize=True) * 100

# 3. Combine into a single detailed DataFrame
comparison_table = pd.DataFrame({
    'Original (count)': orig_counts,
    'Original (%)': orig_pcts,
    'Small Test Set (count)': small_test_counts,
    'Small Test Set (%)': small_test_pcts,
    'Med Test Set (count)': med_test_dist,
    'Med Test Set (%)': med_test_pcts,
    'Large Test Set (count)': large_test_counts,
    'Large Test Set (%)': large_test_pcts
})


labels = encoders['class1'].classes_
comparison_table.index = [labels[i] for i in comparison_table.index]

# 5. Clean up the display
print("--- Detailed Distribution Table (class1) ---")
print(comparison_table.round(5))